In [0]:
import pandas as p
from pyspark.sql import functions as f

# load the cleaned silver table
df_silver = spark.table('data_engineering.video1.users_cleaned')

In [0]:
# extract the day of the week and analyse the signup patterns
df_day_analysis = df_silver.withColumn('day_of_week', f.dayofweek('signup_date')) \
    .withColumn('day_name', f.date_format('signup_Date', 'EEEE')) \
    .groupBy('day_of_week', 'day_name') \
    .agg(
        f.count('user_id').alias('total_signups')
    ) \
    .orderBy(f.desc('total_signups'))

display(df_day_analysis)

In [0]:
# analyse which referral sources were clicked the most
df_referral_analysis = df_silver.groupBy('referral_source') \
    .agg(
        f.count('user_id').alias('total_clicks'),
        f.countDistinct('country').alias('countries_reached')
    ) \
    .orderBy(f.desc('total_clicks'))

display(df_referral_analysis)

In [0]:
# create a comprehensive insights table combining day of week and referral source
df_insights = df_silver.withColumn('day_of_week', f.dayofweek('signup_date')) \
    .withColumn('day_name', f.date_format('signup_Date', 'EEEE')) \
    .groupBy('day_name','referral_source') \
    .agg(
        f.count('user_id').alias('signups'),
        f.countDistinct('country').alias('unique_countries')
    ) \
    .orderBy(f.desc('signups'))

# write to gold table in the same schema
df_insights.write.mode('overwrite').saveAsTable('data_engineering.video1.users_gold')

display(df_insights)